# Session 8: Does the single-night case study generalise?

**Question:** do the four Session 4–7 findings hold across many nights, or were they features of 2022-03-01?

**Builds on:** Sessions 4–7 studied one flight on one night. Their findings: (1) avoidance has steeply diminishing returns; (2) aircraft mass decides whether there is a trade-off at all; (3) uncertainty in the height of the moist layer is what changes route choices; (4) a robust choice is cheap when the aircraft can climb.

**Approach:**
1. Repeat the Session 7 analysis unchanged on 24 nights, chosen by a rule fixed before any results were seen: the 1st and 15th of every month in 2022, departing 00:00 UTC.
2. Check first that the pipeline reproduces Session 7 exactly on 2022-03-01.
3. For each night: contrail exposure on all 45 routes, fuel for four aircraft masses, and the deterministic vs robust route choice under the seven vertical-shift scenarios.
4. Summarise each finding across nights.

**Assumptions:** as Session 7 (same routes, A320, 230 m/s, no wind, cruise only, ERA5 on 10 pressure levels with no humidity scaling). Only the date changes. Humidity, horizontal and timing scenarios are not repeated: in Sessions 5–6 they rarely changed decisions.

**Sections:** 1. Setup · 2. Evaluating one night · 3. Reproduction check (2022-03-01) · 4. All 24 nights · 5. How often is there contrail risk? · 6. Cost of avoidance · 7. Robust choice under vertical uncertainty · 8. Figures · Results

## 1. Setup

The 24 dates and the case definition come from `contrail_routing.config`; the ERA5 data was downloaded in advance by `scripts/download_era5_dates.py` (log in `results/session8/era5_download_log.csv`).

In [10]:
import sys, io, gc, time, warnings, contextlib
from pathlib import Path
SRC = (Path("..") / "src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from contrail_routing import config
from contrail_routing.met import load_era5_checked, load_region
from contrail_routing.scenarios import shift_field, coverage, run_scenarios, pcr_by_scenario
from contrail_routing.performance import fuel_table
from contrail_routing.decisions import choose

warnings.filterwarnings("ignore", message=r"\s*Met data appears to have originated from ECMWF")
pd.set_option("display.width", 220)

results_dir = Path("..") / "results"
fig_dir = results_dir / "figures"
s8_dir = results_dir / "session8"
cache_dir = s8_dir / "cache"     # along-track tables per night (large; not committed)
cache_dir.mkdir(parents=True, exist_ok=True)

DATES = config.STUDY_DATES
FLIGHT_LEVELS = config.FLIGHT_LEVELS + [390]
specs_all = config.route_specs(flight_levels=FLIGHT_LEVELS)
names_v = [n for n, s in specs_all.items() if s["flight_level"] >= 340]   # robust-choice candidates (Session 7)
VERTICAL_M = {"base": 0.0, "up150": 150.0, "up300": 300.0, "up600": 600.0,
              "down150": -150.0, "down300": -300.0, "down600": -600.0}
MASSES = {"light_56t": 56000.0, "model_default": None,
          "heavy_68t": 68000.0, "very_heavy_72t": 72000.0}
LAMBDAS = [0.1, 1.0, 10.0]
print(f"{len(DATES)} nights: {DATES[0]} ... {DATES[-1]}")
print(f"{len(specs_all)} routes, {len(names_v)} robust-choice candidates (FL340-FL390), "
      f"{len(VERTICAL_M)} vertical scenarios, {len(MASSES)} masses")

24 nights: 2022-01-01 ... 2022-12-15
45 routes, 30 robust-choice candidates (FL340-FL390), 7 vertical scenarios, 4 masses


## 2. Evaluating one night

The same steps as Session 7, wrapped in one function: load and check ERA5 (00:00–06:00 UTC), cut out the region, build the routes for that night's departure, check every route lies inside the data, evaluate PCR on all 45 routes and on the 30 FL340–FL390 routes under each vertical shift, and compute fuel for each mass. Along-track results are saved per night, so a re-run reloads them.

In [11]:
def run_night(date):
    """PCR (base and vertical shifts) and fuel for one night."""
    met, report = load_era5_checked((f"{date}T00", f"{date}T06"), config.LEVELS_FINE,
                                    config.REGION_LON, config.REGION_LAT)
    region = load_region(met, config.REGION_LON, config.REGION_LAT)
    met.data.close()
    del met

    routes = config.build_routes(specs_all, departure=pd.Timestamp(f"{date} 00:00"))
    routes_v = {n: routes[n] for n in names_v}
    assert coverage(region, routes)["ok"].all(), f"{date}: routes outside the data"
    scen = {name: shift_field(region, up_m=dz) for name, dz in VERTICAL_M.items()}
    for name, m in scen.items():
        assert coverage(m, routes_v)["ok"].all(), f"{date} {name}: routes outside the shifted data"

    with contextlib.redirect_stdout(io.StringIO()):     # silence per-scenario progress lines
        base = run_scenarios({"base": region}, routes, cache_dir / date, "all")
        vert = run_scenarios(scen, routes_v, cache_dir / date, "vertical")
        fuel = fuel_table(base["base"], config.AIRCRAFT, config.CRUISE_SPEED_MS, MASSES)
    pcr_base = pcr_by_scenario(base, specs_all)["base"]
    pcr_v = pcr_by_scenario(vert, specs_all)
    assert np.allclose(pcr_v["base"], pcr_base[pcr_v.index]), f"{date}: unshifted scenario differs from base"
    fuel["feasible"] = fuel["within_envelope"] & fuel["fuel_kg"].notna()
    return pcr_base, pcr_v, fuel

## 3. Reproduction check (2022-03-01)

On the Session 7 date, the new code must give exactly the Session 7 contrail exposure (base and vertical shifts) and fuel. This confirms that the only thing that changes between nights is the weather.

In [12]:
t0 = time.time()
pcr_b, pcr_v, fuel_d = run_night("2022-03-01")
print(f"2022-03-01 evaluated in {time.time() - t0:.0f} s")

s7_base = pd.read_csv(results_dir / "session7_pcr_coarse_vs_fine.csv", index_col="route")["fine"]
s7_vert = pd.read_csv(results_dir / "session7_pcr_vertical_shifts_v2.csv", index_col="route")
s7_fuel = pd.read_csv(results_dir / "session7_fuel_by_mass.csv").set_index(["route", "mass_case"])["fuel_kg"]
f8 = fuel_d.set_index(["route", "mass_case"])["fuel_kg"]

d_base = (pcr_b - s7_base.reindex(pcr_b.index)).abs().max()
d_vert = (pcr_v - s7_vert.reindex(index=pcr_v.index, columns=pcr_v.columns)).abs().max().max()
d_fuel = (f8 - s7_fuel.reindex(f8.index)).abs().max()
print(f"Max difference from Session 7: PCR base {d_base:.6f} km | vertical {d_vert:.6f} km | fuel {d_fuel:.6f} kg")
assert d_base < 1e-6 and d_vert < 1e-6 and d_fuel < 1e-3   # fuel within 1 g: Session 7 re-read its tracks from CSV (round-off ~1e-6 kg)
print("2022-03-01 reproduces Session 7.")

ERA5 [175, 190, 200, 210, 220, 230, 240, 250, 275, 300] hPa: all 7 hours complete.
2022-03-01 evaluated in 5 s
Max difference from Session 7: PCR base 0.000000 km | vertical 0.000000 km | fuel 0.000000 kg
2022-03-01 reproduces Session 7.


## 4. All 24 nights

Expect roughly a minute per night on the first run; re-runs reload the saved along-track tables. Each line also confirms that night's ERA5 check.

In [ ]:
pcr_base_all, pcr_v_all, fuel_all = {}, {}, {}
t_start = time.time()
for i, date in enumerate(DATES, 1):
    t0 = time.time()
    pcr_base_all[date], pcr_v_all[date], fuel_all[date] = run_night(date)
    gc.collect()
    print(f"[{i:2d}/{len(DATES)}] {date} done in {time.time() - t0:3.0f} s | "
          f"routes with PCR: {int((pcr_base_all[date] > 0).sum()):2d} of {len(specs_all)}")
print(f"Total {(time.time() - t_start) / 60:.1f} min")

pcr_base = pd.DataFrame(pcr_base_all)              # route x date
pcr_vert = (pd.concat({d: v.stack() for d, v in pcr_v_all.items()}, names=["date", "route", "scenario"])
            .rename("pcr_km").reset_index())
fuel = pd.concat(fuel_all, names=["date", "row"]).reset_index(level="row", drop=True).reset_index()
print("Missing PCR values:", int(pcr_base.isna().sum().sum()) + int(pcr_vert["pcr_km"].isna().sum()))
print("Routes x masses outside the flight envelope:",
      fuel.groupby("mass_case")["feasible"].apply(lambda s: int((~s).sum())).to_dict())

pcr_base.to_csv(s8_dir / "pcr_base_by_night.csv", index_label="route")
pcr_vert.to_csv(s8_dir / "pcr_vertical_by_night.csv", index=False)
fuel.to_csv(s8_dir / "fuel_by_night.csv", index=False)

ERA5 [175, 190, 200, 210, 220, 230, 240, 250, 275, 300] hPa: all 7 hours complete.
[ 1/24] 2022-01-01 done in   4 s | routes with PCR: 45 of 45


## 5. How often is there contrail risk?

Exposure on the great-circle route by flight level and night, then the minimum-fuel route for each mass and its exposure. Finding 2 (mass decides whether there is a trade-off) predicts that heavier aircraft, cruising lower, meet persistent-contrail conditions more often.

In [ ]:
gc_fl = pcr_base.loc[[f"GC0_FL{fl}" for fl in FLIGHT_LEVELS]]
gc_fl.index = pd.Index(FLIGHT_LEVELS, name="FL")
print("Great-circle route: km in PCR conditions (rows = nights, columns = flight level)")
print(gc_fl.T.round(0).astype(int).to_string())
print(f"\nNights with PCR on at least one of the 45 routes: {int((pcr_base > 0).any().sum())} of {len(DATES)}")
print("Nights with PCR on the great circle, by flight level:", (gc_fl > 0).sum(axis=1).to_dict())

rows = []
for (date, mc), g in fuel[fuel["feasible"]].groupby(["date", "mass_case"], sort=False):
    g = g.set_index("route")
    r = g["fuel_kg"].idxmin()
    pv = pcr_v_all[date]
    rows.append({"date": date, "mass_case": mc, "min_fuel_route": r,
                 "flight_level": specs_all[r]["flight_level"], "fuel_kg": g.at[r, "fuel_kg"],
                 "pcr_base_km": pcr_base.at[r, date],
                 "pcr_worst_vertical_km": pv.loc[r].max() if r in pv.index else np.nan})
min_fuel = pd.DataFrame(rows)
exp_min = min_fuel.pivot(index="date", columns="mass_case", values="pcr_base_km")[list(MASSES)]
lvl_min = min_fuel.pivot(index="date", columns="mass_case", values="flight_level")[list(MASSES)]
print("\nMinimum-fuel flight level by mass:")
print(lvl_min.apply(lambda s: s.value_counts().sort_index().to_dict()).to_string())
print("\nMinimum-fuel route: km in PCR conditions (base)")
print(exp_min.round(0).to_string())
print("\nNights where the minimum-fuel route has PCR exposure:", (exp_min > 0).sum().to_dict())
worst_min = min_fuel.pivot(index="date", columns="mass_case", values="pcr_worst_vertical_km")[list(MASSES)]
print("...and where it has exposure in at least one vertical scenario:", (worst_min > 0).sum().to_dict())
min_fuel.to_csv(s8_dir / "min_fuel_route_by_night.csv", index=False)

## 6. Cost of avoidance

For each night with exposure on the minimum-fuel route (model-default mass), the cheapest route that cuts exposure by at least 80%, and the cheapest route with none, both as extra fuel. Done without a ceiling and with the aircraft held at or below FL360, the case behind finding 1 (83% cut for +1.6% on 2022-03-01).

In [ ]:
def avoidance_cost(date, mass_case="model_default", max_fl=None):
    f = fuel[(fuel["date"] == date) & (fuel["mass_case"] == mass_case) & fuel["feasible"]].set_index("route")
    if max_fl is not None:
        f = f[[specs_all[r]["flight_level"] <= max_fl for r in f.index]]
    m = pd.DataFrame({"fuel_kg": f["fuel_kg"], "pcr_km": pcr_base.loc[f.index, date]})
    ref = m["fuel_kg"].idxmin()
    ref_fuel, ref_pcr = m.at[ref, "fuel_kg"], m.at[ref, "pcr_km"]
    row = {"date": date, "ceiling": "none" if max_fl is None else f"FL{max_fl}",
           "ref_route": ref, "ref_pcr_km": ref_pcr}
    for label, ok in [("cut80", m["pcr_km"] <= 0.2 * ref_pcr), ("zero", m["pcr_km"] == 0)]:
        if ref_pcr == 0 or not ok.any():
            row[f"{label}_route"], row[f"{label}_extra_pct"] = (ref, 0.0) if ref_pcr == 0 else (None, np.nan)
            continue
        r = m.loc[ok, "fuel_kg"].idxmin()
        row[f"{label}_route"] = r
        row[f"{label}_extra_pct"] = 100 * (m.at[r, "fuel_kg"] - ref_fuel) / ref_fuel
    return row

avoid = pd.DataFrame([avoidance_cost(d, max_fl=c) for c in [None, 360] for d in DATES])
for c, g in avoid.groupby("ceiling", sort=False):
    exposed = g[g["ref_pcr_km"] > 0]
    print(f"\nCeiling {c}: {len(exposed)} of {len(g)} nights with exposure on the minimum-fuel route")
    if len(exposed):
        print(exposed[["date", "ref_route", "ref_pcr_km", "cut80_route", "cut80_extra_pct",
                       "zero_route", "zero_extra_pct"]].round(2).to_string(index=False))
        print("Extra fuel (%) for an 80% cut: median {:.2f}, max {:.2f}; for zero exposure: median {:.2f}, max {:.2f}, "
              "not possible on {} nights".format(exposed["cut80_extra_pct"].median(), exposed["cut80_extra_pct"].max(),
                                                exposed["zero_extra_pct"].median(), exposed["zero_extra_pct"].max(),
                                                int(exposed["zero_extra_pct"].isna().sum())))
avoid.to_csv(s8_dir / "avoidance_cost_by_night.csv", index=False)

## 7. Robust choice under vertical uncertainty

The Session 7 analysis for every night: for each mass and λ, the deterministic, worst-case and minimum-regret rules choose among the feasible FL340–FL390 routes across the seven vertical scenarios. On 2022-03-01 the rules disagreed in 9 of 12 cases.

In [ ]:
decisions, summary = [], []
for date in DATES:
    pv = pcr_v_all[date]
    fd = fuel[fuel["date"] == date]
    fuel_by_mass = fd.pivot(index="route", columns="mass_case", values="fuel_kg")
    feas = fd.pivot(index="route", columns="mass_case", values="feasible").astype(bool)
    for mc in MASSES:
        f = fuel_by_mass[mc]
        cands = [r for r in names_v if feas.at[r, mc]]
        min_cand = f[cands].idxmin()
        clean = [r for r in cands if pv.loc[r].max() == 0]
        best_clean = min(clean, key=lambda r: f[r]) if clean else None
        summary.append({
            "date": date, "mass_case": mc, "min_fuel_route": min_cand,
            "its_pcr_base_km": pv.at[min_cand, "base"], "its_pcr_worst_km": pv.loc[min_cand].max(),
            "cheapest_always_clean": best_clean,
            "extra_fuel_always_clean_pct": 100 * (f[best_clean] - f[min_cand]) / f[min_cand] if best_clean else np.nan,
        })
        for lam in LAMBDAS:
            d = choose(pv, f, cands, lam)
            d.insert(0, "lambda", lam)
            d.insert(0, "mass_case", mc)
            d.insert(0, "date", date)
            decisions.append(d)
summary = pd.DataFrame(summary)
decisions = pd.concat(decisions, ignore_index=True)

case_keys = ["date", "mass_case", "lambda"]
disagree = decisions.groupby(case_keys)["route"].nunique() > 1
print(f"Cases where the rules disagree: {int(disagree.sum())} of {len(disagree)}")
print("\nBy mass and lambda (number of nights out of 24):")
print(disagree.groupby(["mass_case", "lambda"]).sum().unstack()[LAMBDAS].loc[list(MASSES)].to_string())
print("\nNights with at least one disagreement:", int(disagree.groupby("date").any().sum()), "of", len(DATES))

wc = decisions[decisions["rule"] == "worst_case"].copy()
wc["extra_pct"] = 100 * wc["extra_fuel_vs_det_kg"] / (wc["fuel_kg"] - wc["extra_fuel_vs_det_kg"])
print("\nExtra fuel of the worst-case choice over the deterministic one (%), lambda = 1:")
print(wc[wc["lambda"] == 1.0].groupby("mass_case")["extra_pct"].describe()[["mean", "50%", "max"]]
      .loc[list(MASSES)].round(2).to_string())

print("\nCheapest route that is contrail-free in every vertical scenario (extra fuel %):")
s = summary.groupby("mass_case")["extra_fuel_always_clean_pct"]
print(pd.DataFrame({"nights_with_one": s.count(), "median": s.median(), "max": s.max()})
      .loc[list(MASSES)].round(2).to_string())
print("\nNights where the minimum-fuel candidate is exposed in some vertical scenario:",
      summary.assign(x=summary["its_pcr_worst_km"] > 0).groupby("mass_case")["x"].sum().loc[list(MASSES)].to_dict())

summary.to_csv(s8_dir / "robust_summary_by_night.csv", index=False)
decisions.to_csv(s8_dir / "robust_decisions_by_night.csv", index=False)

## 8. Figures

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
im = ax.imshow(gc_fl.values, aspect="auto", cmap="Blues", origin="lower")
ax.set_xticks(range(len(DATES)))
ax.set_xticklabels([d[5:] for d in DATES], rotation=90)
ax.set_yticks(range(len(FLIGHT_LEVELS)))
ax.set_yticklabels([f"FL{fl}" for fl in FLIGHT_LEVELS])
ax.set_xlabel("Night in 2022 (month-day), departure 00:00 UTC")
ax.set_title("Great-circle route: distance flown in persistent-contrail conditions")
fig.colorbar(im, ax=ax, label="km in PCR conditions")
fig.savefig(fig_dir / "session8_exposure_by_night.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for k, mc in enumerate(MASSES):
    y = exp_min[mc].values
    axes[0].scatter(np.full(len(y), k) + np.linspace(-0.25, 0.25, len(y)), y, s=18)
    v = summary.loc[summary["mass_case"] == mc, "extra_fuel_always_clean_pct"].values
    ok = ~np.isnan(v)
    x = np.full(len(v), k) + np.linspace(-0.25, 0.25, len(v))
    axes[1].scatter(x[ok], v[ok], s=18)
    axes[1].scatter(x[~ok], np.full((~ok).sum(), np.nanmax(summary["extra_fuel_always_clean_pct"].values) * 1.15 + 0.1),
                    marker="x", color="k", s=25)
for ax in axes:
    ax.set_xticks(range(len(MASSES)))
    ax.set_xticklabels(list(MASSES))
    ax.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("km in PCR conditions")
axes[0].set_title("Minimum-fuel route: exposure on each night")
axes[1].set_ylabel("extra fuel (%)")
axes[1].set_title("Cheapest route clean in every vertical scenario\n(x = none exists that night)")
fig.savefig(fig_dir / "session8_mass_and_robust_cost.png", dpi=150, bbox_inches="tight")
plt.show()

## Results

### Setup

The Session 7 analysis repeated unchanged on 24 nights, the 1st and 15th of every month in 2022 (rule fixed before any results were seen), departing 00:00 UTC: 45 routes (5 lateral offsets × FL310–FL390) from (−10°, 45°) to (10°, 55°), A320, 230 m/s, no wind, cruise only, ERA5 on 10 pressure levels (00:00–06:00 UTC) with no humidity scaling. Fuel for four start-of-cruise masses (56 t, model estimate ≈64 t, 68 t, 72 t). Deterministic, worst-case and minimum-regret route choice among the feasible FL340–FL390 routes across seven vertical shifts of the weather field (0, ±150, ±300, ±600 m), for λ = 0.1, 1 and 10 kg/km.

### Checks

- ERA5 for all 24 nights was downloaded in advance by `scripts/download_era5_dates.py`: every hour checked, no missing values, no repairs needed (`results/session8/era5_download_log.csv`).
- 2022-03-01 reproduces Session 7: contrail exposure identical (base and all vertical scenarios); fuel within 6 × 10⁻⁶ kg (rounding from Session 7 re-reading its tracks from CSV).
- Every route lies inside the data for every night and every vertical scenario; no missing values in any result.
- 20 automated tests pass, including two new ones for the date definition.
- The 72 t aircraft is outside the flight envelope at FL390 on every night (5 routes × 24 nights = 120 cases), as in Session 7; every other mass and route is feasible.

### Findings

#### Contrail risk is common at cruise levels

Every one of the 24 nights had persistent-contrail (PCR) conditions on at least one of the 45 routes. On the great circle, FL330–FL350 were exposed on 19 nights, FL370 on 11, FL380 on 8 and FL390 on 6. The height of the moist layer varies from night to night: on all six summer nights exposure reached FL370 or higher, on no spring night did it rise above FL360, and winter and autumn were mixed (1 January and 1 February were exposed up to FL390).

#### Aircraft mass (finding 2): holds

| Start-of-cruise mass | Minimum-fuel level (nights) | Minimum-fuel route exposed | ...in at least one vertical scenario |
|---|---|---|---|
| 56 t | FL390 (24) | 6 of 24 | 11 of 24 |
| model estimate (≈64 t) | FL370 (17), FL380 (7) | 11 of 24 | 17 of 24 |
| 68 t | FL370 (20), FL360 (4) | 12 of 24 | 19 of 24 |
| 72 t | FL350 (15), FL360 (9) | 17 of 24 | 19 of 24 |

Heavier aircraft cruise lower and meet PCR conditions more often. On 2022-03-01 only the 72 t aircraft was exposed; across nights, the model-estimate mass is exposed on almost half of them.

#### Cost of avoidance (finding 1): the single-night figure was optimistic

Model-estimate mass; extra fuel relative to the minimum-fuel route on nights when that route is exposed.

| | No ceiling | Held at or below FL360 |
|---|---|---|
| Nights with exposure on the minimum-fuel route | 11 of 24 | 15 of 24 |
| ≥80% cut: median / max extra fuel | 1.23% / 4.82% | 4.33% / 13.10% |
| ≥80% cut not possible | 1 night | 3 nights |
| Zero exposure not possible | 1 night | 5 nights |
| 2022-03-01 | no exposure | 1.59% (NW150_FL360) |

Without a ceiling, avoidance is often cheap (one level up: 0.05–0.13% on three nights) but sometimes costs 2–5%. Under the FL360 ceiling, the 1.6% found on 2022-03-01 is at the cheap end; the typical cost is about 4%.

#### Robust choice under vertical uncertainty (findings 3 and 4)

- The three decision rules chose different routes in **142 of 288 cases (49%)**, on **19 of 24 nights**; on 2022-03-01 alone it was 9 of 12. Disagreements rise with mass and λ (at λ = 10: 9, 15, 18 and 18 nights for 56, ≈64, 68 and 72 t).
- The worst-case choice costs nothing extra on a median night for every mass (λ = 1); the mean is 0.13–0.39% and the maximum 1.64–6.28%.
- A candidate that stays contrail-free in every vertical scenario exists on only 16 of 24 nights (13 for 72 t). When it exists, its median extra fuel is 0.00%, 0.03%, 0.46% and 1.50% for the four masses, and up to about 6.8%.

### Interpretation

2022-03-01 was a fairly favourable night: the moist layer ended just below the efficient cruise levels, so light and medium aircraft were clear and a robust choice was nearly free. Across 24 nights the qualitative picture survives: mass decides how often there is a trade-off, and uncertainty in the height of the moist layer changes route choices on most nights. The costs, however, vary widely between nights: avoidance and robustness are cheap on a typical night but can cost several percent of fuel, and on about a third of nights no route in this set is safe against every vertical error. A planner that searches beyond this fixed route set, and a calibrated estimate of how large vertical errors really are, are the natural next steps.

### Corrections

- The reproduction check first required fuel to match Session 7 within 10⁻⁶ kg, tighter than the rounding introduced by Session 7 saving and re-reading its tracks (6 × 10⁻⁶ kg on ≈5,000 kg). The fuel tolerance was set to 1 g; contrail exposure matched exactly.
- 2022-01-01 appears twice in the download log: the first run downloaded 00:00–04:00, and the time window was then extended to 00:00–06:00 to match Session 7.
- The first version of this summary said winter and spring nights were exposed mainly below FL350. The data shows four spring nights exposed at FL360 and 1 February up to FL390; the sentence was corrected.

### Limitations

- 24 nights from one year, one departure time (00:00 UTC) and one city pair; the sample is systematic, not random, and too small for firm seasonal conclusions.
- Only vertical-shift scenarios were repeated; humidity, horizontal and timing errors were not.
- The vertical shift sizes (150–600 m) are still sensitivity cases, not estimates of ERA5 error.
- The disagreement count depends on the chosen λ values and on the fixed route set.
- Cruise only, constant true airspeed, no wind; PCR distance is a proxy for climate impact.